In [1]:
# Configure notebook environment
from src.utils.project_setup import configure_notebook

configure_notebook()

# General libraries
from pathlib import Path
import pandas as pd
import numpy as np

# Optimization
import pulp

# Visualization
import plotly.express as px
import plotly.graph_objects as go

Notebook configured successfully.


In [2]:
# Load XGBoost test-period forecasts
forecast_path = Path("../outputs/predictions/final_forecast.parquet")

forecast_data = pd.read_parquet(forecast_path)

print(f"Forecast dataset shape: {forecast_data.shape}")
forecast_data.head()

Forecast dataset shape: (853720, 2)


,Actual,Forecast
0,2,0.800423
1,1,0.766709
2,1,0.789739
3,0,0.812287
4,4,0.977677


In [3]:
# Load engineered features
from src.data.loader import load_features

model_data = load_features()

# Use the same 365-day window as Notebook 06
latest_date = model_data["date"].max()
cutoff_date = latest_date - pd.Timedelta(days=365)

model_data = model_data[
    model_data["date"] >= cutoff_date
].reset_index(drop=True)

# Recreate the chronological test period
dates = np.sort(model_data["date"].unique())

validation_end = dates[-28]

test_data = model_data[
    model_data["date"] >= validation_end
].copy()

print(f"Test-period records: {test_data.shape}")

Test-period records: (853720, 38)


In [4]:
# Build optimization input data
optimization_data = test_data[
    [
        "id",
        "item_id",
        "store_id",
        "state_id",
        "date",
        "sales",
        "sell_price",
    ]
].reset_index(drop=True)

# Add forecasts from Notebook 06
optimization_data["forecast_demand"] = forecast_data["Forecast"].values

# Keep non-negative demand forecasts
optimization_data["forecast_demand"] = (
    optimization_data["forecast_demand"]
    .clip(lower=0)
)

optimization_data.head()

,id,item_id,store_id,state_id,date,sales,sell_price,forecast_demand
0,FOODS_1_001_CA_1_validation,FOODS_1_001,CA_1,CA,2016-03-28,2,2.24,0.800423
1,FOODS_1_001_CA_1_validation,FOODS_1_001,CA_1,CA,2016-03-29,1,2.24,0.766709
2,FOODS_1_001_CA_1_validation,FOODS_1_001,CA_1,CA,2016-03-30,1,2.24,0.789739
3,FOODS_1_001_CA_1_validation,FOODS_1_001,CA_1,CA,2016-03-31,0,2.24,0.812287
4,FOODS_1_001_CA_1_validation,FOODS_1_001,CA_1,CA,2016-04-01,4,2.24,0.977677


In [5]:
# Aggregate forecast demand across the test period
inventory_input = (
    optimization_data
    .groupby(
        ["item_id", "store_id", "state_id"],
        as_index=False
    )
    .agg(
        forecast_demand=("forecast_demand", "sum"),
        actual_demand=("sales", "sum"),
        average_price=("sell_price", "mean"),
    )
)

print(f"Item-store decisions: {inventory_input.shape}")
inventory_input.head()

Item-store decisions: (30490, 6)


,item_id,store_id,state_id,forecast_demand,actual_demand,average_price
0,FOODS_1_001,CA_1,CA,31.751419,33,2.24
1,FOODS_1_001,CA_2,CA,25.423388,31,2.24
2,FOODS_1_001,CA_3,CA,32.444618,24,2.24
3,FOODS_1_001,CA_4,CA,11.592309,9,2.24
4,FOODS_1_001,TX_1,TX,9.201897,1,2.24


In [6]:
# Select a representative optimization sample
optimization_sample = (
    inventory_input
    .sample(
        n=min(100, len(inventory_input)),
        random_state=42
    )
    .reset_index(drop=True)
)

optimization_sample.head()

,item_id,store_id,state_id,forecast_demand,actual_demand,average_price
0,FOODS_2_025,TX_1,TX,10.519777,11,3.86
1,FOODS_3_145,TX_3,TX,7.243583,6,2.98
2,FOODS_3_261,CA_2,CA,42.509094,44,3.98
3,FOODS_2_261,TX_3,TX,13.823271,15,9.88
4,HOBBIES_1_118,WI_3,WI,4.516345,5,11.12


In [30]:
# Inventory optimization assumptions

HOLDING_COST_RATE = 0.15
STOCKOUT_COST_MULTIPLIER = 2.0

# Safety stock protects against forecast uncertainty
SAFETY_STOCK_RATE = 0.20

# Capacity is intentionally constrained to force trade-offs
CAPACITY_RATE = 0.95

optimization_sample["holding_cost"] = (
    optimization_sample["average_price"] * HOLDING_COST_RATE
)

optimization_sample["stockout_cost"] = (
    optimization_sample["average_price"] * STOCKOUT_COST_MULTIPLIER
)

optimization_sample["safety_stock"] = (
    optimization_sample["forecast_demand"] * SAFETY_STOCK_RATE
)

optimization_sample["target_inventory"] = (
    optimization_sample["forecast_demand"]
    + optimization_sample["safety_stock"]
)

WAREHOUSE_CAPACITY = (
    optimization_sample["target_inventory"].sum()
    * CAPACITY_RATE
)

print(f"Total forecast demand: {optimization_sample['forecast_demand'].sum():,.0f}")
print(f"Total target inventory: {optimization_sample['target_inventory'].sum():,.0f}")
print(f"Warehouse capacity: {WAREHOUSE_CAPACITY:,.0f}")

Total forecast demand: 3,081
Total target inventory: 3,697
Warehouse capacity: 3,512


In [32]:
# Create a linear optimization model
model = pulp.LpProblem(
    "Inventory_Allocation_Optimization",
    pulp.LpMinimize
)

# Index each item-store decision
items = optimization_sample.index.tolist()

# Decision variables
inventory = pulp.LpVariable.dicts(
    "inventory",
    items,
    lowBound=0,
    cat="Continuous"
)

shortage = pulp.LpVariable.dicts(
    "shortage",
    items,
    lowBound=0,
    cat="Continuous"
)

excess = pulp.LpVariable.dicts(
    "excess",
    items,
    upBound=10,
    cat="Continuous"
)

In [33]:
# Minimize holding and stockout costs
model += pulp.lpSum(
    optimization_sample.loc[i, "holding_cost"] * excess[i]
    + optimization_sample.loc[i, "stockout_cost"] * shortage[i]
    for i in items
)

In [34]:
# Demand balance and inventory target constraints

for i in items:
    target = optimization_sample.loc[i, "target_inventory"]

    # Inventory plus shortage must cover the target inventory level
    model += inventory[i] + shortage[i] >= target

    # Excess represents inventory above the target
    model += excess[i] >= inventory[i] - target

# Total inventory capacity constraint
model += pulp.lpSum(
    inventory[i] for i in items
) <= WAREHOUSE_CAPACITY

In [35]:
# Solve using CBC, the default open-source PuLP solver
model.solve(pulp.PULP_CBC_CMD(msg=False))

print(f"Solver status: {pulp.LpStatus[model.status]}")
print(f"Optimized total cost: ${pulp.value(model.objective):,.2f}")

Solver status: Optimal
Optimized total cost: $267.00


In [36]:
optimization_sample["recommended_inventory"] = [
    inventory[i].value()
    for i in items
]

optimization_sample["expected_shortage"] = [
    shortage[i].value()
    for i in items
]

optimization_sample["expected_excess"] = [
    excess[i].value()
    for i in items
]

optimization_results = optimization_sample[
    [
        "item_id",
        "store_id",
        "state_id",
        "forecast_demand",
        "safety_stock",
        "target_inventory",
        "actual_demand",
        "recommended_inventory",
        "expected_shortage",
        "expected_excess",
        "average_price",
        "holding_cost",
        "stockout_cost",
    ]
]

optimization_results.head(20)

,item_id,store_id,state_id,forecast_demand,safety_stock,target_inventory,actual_demand,recommended_inventory,expected_shortage,expected_excess,average_price,holding_cost,stockout_cost
0,FOODS_2_025,TX_1,TX,10.519777,2.103956,12.623733,11,12.623733,0.0,0.0,3.86,0.5790,7.72
1,FOODS_3_145,TX_3,TX,7.243583,1.448717,8.692300,6,8.692300,0.0,0.0,2.98,0.4470,5.96
2,FOODS_3_261,CA_2,CA,42.509094,8.501819,51.010914,44,51.010914,0.0,0.0,3.98,0.5970,7.96
3,FOODS_2_261,TX_3,TX,13.823271,2.764654,16.587925,15,16.587925,0.0,0.0,9.88,1.4820,19.76
4,HOBBIES_1_118,WI_3,WI,4.516345,0.903269,5.419614,5,5.419614,0.0,0.0,11.12,1.6680,22.24
5,FOODS_3_365,CA_4,CA,20.553940,4.110788,24.664728,22,24.664728,0.0,0.0,5.24,0.7860,10.48
6,HOBBIES_1_315,WI_3,WI,1.887964,0.377593,2.265556,2,2.265556,0.0,0.0,4.97,0.7455,9.94
7,HOBBIES_2_072,TX_1,TX,0.757848,0.151570,0.909418,0,0.909418,0.0,0.0,7.97,1.1955,15.94
8,FOODS_1_112,CA_3,CA,31.998837,6.399767,38.398605,30,38.398605,0.0,0.0,1.98,0.2970,3.96
9,HOBBIES_1_246,CA_1,CA,1.112382,0.222476,1.334858,2,1.334858,0.0,0.0,10.98,1.6470,21.96


In [37]:
# Evaluate the recommended inventory plan using actual demand
optimization_results["actual_shortage"] = (
    optimization_results["actual_demand"]
    - optimization_results["recommended_inventory"]
).clip(lower=0)

optimization_results["actual_excess"] = (
    optimization_results["recommended_inventory"]
    - optimization_results["actual_demand"]
).clip(lower=0)

total_actual_demand = optimization_results["actual_demand"].sum()
total_actual_shortage = optimization_results["actual_shortage"].sum()

service_level = 1 - (
    total_actual_shortage / total_actual_demand
)

print(f"Total actual demand: {total_actual_demand:,.0f}")
print(f"Total actual shortage: {total_actual_shortage:,.0f}")
print(f"Estimated service level: {service_level:.2%}")

Total actual demand: 3,167
Total actual shortage: 205
Estimated service level: 93.52%


In [38]:
optimization_results["inventory_gap_vs_forecast"] = (
    optimization_results["recommended_inventory"]
    - optimization_results["forecast_demand"]
)

optimization_results["inventory_gap_vs_target"] = (
    optimization_results["recommended_inventory"]
    - optimization_results["target_inventory"]
)

optimization_results.sort_values(
    "expected_shortage",
    ascending=False
).head(20)

,item_id,store_id,state_id,forecast_demand,safety_stock,target_inventory,actual_demand,recommended_inventory,expected_shortage,expected_excess,average_price,holding_cost,stockout_cost,actual_shortage,actual_excess,inventory_gap_vs_forecast,inventory_gap_vs_target
20,HOBBIES_1_261,TX_1,TX,56.282688,11.256537,67.539223,68,0.000000,67.539223,-67.539223,0.48,0.0720,0.96,68.000000,0.000000,-56.282688,-6.753922e+01
56,HOUSEHOLD_2_342,TX_2,TX,52.785206,10.557041,63.342247,60,0.000000,63.342247,-63.342247,0.97,0.1455,1.94,60.000000,0.000000,-52.785206,-6.334225e+01
95,HOUSEHOLD_1_327,CA_3,CA,207.284470,41.456894,248.741364,250,216.097090,32.644271,-32.644271,0.98,0.1470,1.96,33.902910,0.000000,8.812620,-3.264427e+01
69,HOUSEHOLD_1_173,WI_1,WI,13.852600,2.770520,16.623119,12,0.000000,16.623119,-16.623119,0.97,0.1455,1.94,12.000000,0.000000,-13.852600,-1.662312e+01
35,HOBBIES_2_017,TX_1,TX,3.915178,0.783036,4.698213,2,0.000000,4.698213,-4.698213,0.50,0.0750,1.00,2.000000,0.000000,-3.915178,-4.698213e+00
3,FOODS_2_261,TX_3,TX,13.823271,2.764654,16.587925,15,16.587925,0.000000,0.000000,9.88,1.4820,19.76,0.000000,1.587925,2.764654,4.272461e-08
4,HOBBIES_1_118,WI_3,WI,4.516345,0.903269,5.419614,5,5.419614,0.000000,0.000000,11.12,1.6680,22.24,0.000000,0.419614,0.903269,-3.819580e-08
5,FOODS_3_365,CA_4,CA,20.553940,4.110788,24.664728,22,24.664728,0.000000,0.000000,5.24,0.7860,10.48,0.000000,2.664728,4.110788,-1.646729e-07
8,FOODS_1_112,CA_3,CA,31.998837,6.399767,38.398605,30,38.398605,0.000000,0.000000,1.98,0.2970,3.96,0.000000,8.398605,6.399768,-3.466797e-07
2,FOODS_3_261,CA_2,CA,42.509094,8.501819,51.010914,44,51.010914,0.000000,0.000000,3.98,0.5970,7.96,0.000000,7.010914,8.501820,1.511230e-07


In [39]:
# Compare forecasts with inventory recommendations
plot_data = optimization_results.head(20).copy()

plot_data["item_store"] = (
    plot_data["item_id"].astype(str)
    + " | "
    + plot_data["store_id"].astype(str)
)

fig = px.bar(
    plot_data,
    x="item_store",
    y=[
        "forecast_demand",
        "recommended_inventory",
        "actual_demand",
    ],
    barmode="group",
    title="Forecast Demand vs Recommended Inventory vs Actual Demand",
    labels={
        "value": "Units",
        "item_store": "Item | Store",
        "variable": "Measure",
    },
)

fig.update_layout(
    xaxis_tickangle=-45
)

fig.show()

The optimizer balances three competing priorities:

1. Cover expected demand.
2. Maintain safety stock for forecast uncertainty.
3. Respect limited warehouse capacity.

As a result, inventory is allocated preferentially toward item-store combinations where stockouts are more expensive, while lower-priority combinations may receive less than their target inventory.

In [40]:
# Saving the optimization input for next notebook
# Save optimization input for scenario analysis
optimization_sample.to_csv(
    "../outputs/optimization_input.csv",
    index=False
)

print("Optimization input saved.")

Optimization input saved.


In [41]:
# Save full item-store optimization input
inventory_input.to_csv(
    "../outputs/full_optimization_input.csv",
    index=False
)

print("Full optimization input saved.")

Full optimization input saved.
